In [6]:
run_date = ""

StatementMeta(, e54ecbee-304d-45c0-b1dc-a26c03200547, 8, Finished, Available, Finished, False)

In [7]:

from datetime import datetime, timezone

effective_date = (
    run_date or datetime.now(timezone.utc).date().isoformat()
)

windrose_dag = {
    "activities": [
        {
            "name": "Bronze",
            "path": "nb_windrose_bronze_ingest",
            "args": {
                "run_date": effective_date,
                "useRootDefaultLakehouse": True
                },
            "timeoutPerCellInSeconds": 300
        },
        {
            "name": "Silver",
            "path": "nb_windrose_silver_transform",
            "args": {
                "run_date": effective_date,
                "useRootDefaultLakehouse": True
                },
            "dependencies": ["Bronze"],
            "timeoutPerCellInSeconds": 300
        },
        {
            "name": "Gold",
            "path": "windrose_gold_transform",
            "args": {
                "useRootDefaultLakehouse": True
            },
            "dependencies": ["Silver"],
            "timeoutPerCellInSeconds": 300
        }
    ],
    "concurrency": 1
}

print(notebookutils.notebook.validateDAG(windrose_dag))




StatementMeta(, e54ecbee-304d-45c0-b1dc-a26c03200547, 9, Finished, Available, Finished, False)

True


In [8]:
# Execute Bronze → Silver → Gold
results = notebookutils.notebook.runMultiple(windrose_dag)

print(results)

StatementMeta(, e54ecbee-304d-45c0-b1dc-a26c03200547, 10, Finished, Available, Finished, False)

{'Bronze': {'exitVal': '', 'exception': None}, 'Silver': {'exitVal': '', 'exception': None}, 'Gold': {'exitVal': '', 'exception': None}}


In [9]:

from pyspark.sql import functions as F

silver = spark.table("dbo.weather_forecast_silver")
gold = spark.table("dbo.weather_daily_gold")

silver_count = silver.count()

duplicate_keys = (
    silver.groupBy("city", "event_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

gold_count = gold.count()

gold_readings = (
    gold.agg(F.sum("hourly_readings").alias("total"))
    .first()["total"] or 0
)

print(f"Silver records: {silver_count}")
print(f"Duplicate Silver keys: {duplicate_keys}")
print(f"Gold daily records: {gold_count}")
print(f"Gold hourly readings: {gold_readings}")

assert duplicate_keys == 0, "Duplicate Silver keys found"
assert silver_count == gold_readings, (
    "Gold aggregation does not account for all Silver records"
)

print("Windrose data validation passed")


StatementMeta(, e54ecbee-304d-45c0-b1dc-a26c03200547, 11, Finished, Available, Finished, False)

Silver records: 96
Duplicate Silver keys: 0
Gold daily records: 4
Gold hourly readings: 96
Windrose data validation passed


In [2]:
from pyspark.sql import functions as F

(
    spark.table("dbo.weather_forecast_silver")
    .groupBy(F.to_date("event_time").alias("forecast_date"))
    .count()
    .orderBy("forecast_date")
    .show()
)

StatementMeta(, 9bd47ec4-742e-41d9-be68-f04bcd1542bd, 4, Finished, Available, Finished, False)

+-------------+-----+
|forecast_date|count|
+-------------+-----+
|   2026-11-07|   24|
|   2026-11-08|   24|
|   2026-11-23|   24|
|   2026-11-24|   24|
+-------------+-----+

